# Solving Linear Inverse Problems Using the Prior Implicit in a Denoiser — Implementation / 구현

**Paper**: Kadkhodaie, Z., & Simoncelli, E. P. (2021). *Proc. NeurIPS*. [arXiv:2007.13640]

## Overview / 개요

이 노트북은 Kadkhodaie-Simoncelli의 핵심 통찰을 검증한다:
1. **Tweedie/Miyasawa identity**: 가우시안 노이즈 디노이저의 잔차는 score $\sigma^2 \nabla_y \log p_\sigma(y)$.
2. **Stochastic ascent**: score 따라 *coarse-to-fine* gradient ascent로 prior에서 high-probability 샘플 추출.
3. **Constrained sampling**: 측정 부분공간으로의 projected gradient를 추가하면 임의 선형 역문제 (inpainting) 풀이.

This notebook verifies Kadkhodaie-Simoncelli's three core ideas in a tiny synthetic setting:
1. **Tweedie/Miyasawa identity** verification on a known Gaussian-mixture prior — denoiser residual = $\sigma^2 \nabla \log p_\sigma$.
2. **Stochastic ascent** (Algorithm 1) draws samples from the implicit prior.
3. **Constrained sampling** (Algorithm 2) solves a toy inpainting problem on a synthetic image.

Uses an *analytic* MMSE denoiser (closed-form for Gaussian mixtures) so we do NOT need a pretrained CNN.

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from numpy.typing import NDArray

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.size'] = 10
rng = np.random.default_rng(42)
print('Modules loaded.')

---

## Part 1: Tweedie/Miyasawa identity verification (1-D Gaussian mixture) / 1차원 GMM에서 Tweedie 항등식 검증

Prior: $p(x) = 0.5\,\mathcal N(-3, 0.3^2) + 0.5\,\mathcal N(+3, 0.3^2)$ (bimodal).

Closed-form MMSE denoiser for Gaussian noise on Gaussian mixture is well-known:
$$
\hat x(y) = \frac{\sum_k w_k\,\frac{\sigma_k^{-2} \mu_k + \sigma^{-2} y}{\sigma_k^{-2} + \sigma^{-2}}\,\mathcal N(y; \mu_k, \sigma_k^2 + \sigma^2)}{\sum_k w_k\,\mathcal N(y; \mu_k, \sigma_k^2 + \sigma^2)}.
$$

The score $\nabla_y \log p_\sigma(y)$ can also be written analytically. We verify $f(y) := \hat x(y) - y = \sigma^2 \nabla_y \log p_\sigma(y)$.

1차원 Gaussian mixture에서 *analytical* MMSE denoiser와 score 계산. Tweedie 항등식 $f(y) = \sigma^2 \nabla_y \log p_\sigma(y)$의 좌·우변이 일치하는지 검증.

In [ ]:
def gmm_pdf(y: NDArray, mus: NDArray, sigmas: NDArray, weights: NDArray) -> NDArray:
    """PDF of a 1-D Gaussian mixture at points y."""
    y = np.atleast_1d(y).astype(np.float64)
    components = (
        weights[None, :] *
        np.exp(-0.5 * ((y[:, None] - mus[None, :]) / sigmas[None, :]) ** 2) /
        (np.sqrt(2 * np.pi) * sigmas[None, :])
    )
    return components.sum(axis=1)


def gmm_mmse_denoiser(y: NDArray, mus: NDArray, sigmas: NDArray, weights: NDArray, noise_std: float) -> NDArray:
    """Analytic MMSE denoiser for 1-D Gaussian mixture observed under Gaussian noise of std noise_std."""
    y = np.atleast_1d(y).astype(np.float64)
    obs_var = sigmas ** 2 + noise_std ** 2
    obs_std = np.sqrt(obs_var)
    log_resp = (
        np.log(weights[None, :])
        - 0.5 * ((y[:, None] - mus[None, :]) ** 2 / obs_var[None, :])
        - np.log(obs_std[None, :])
    )
    log_resp -= log_resp.max(axis=1, keepdims=True)
    resp = np.exp(log_resp)
    resp /= resp.sum(axis=1, keepdims=True)
    posterior_mean = (sigmas ** 2 / obs_var)[None, :] * y[:, None] + (noise_std ** 2 / obs_var * mus)[None, :]
    return (resp * posterior_mean).sum(axis=1)


def gmm_score(y: NDArray, mus: NDArray, sigmas: NDArray, weights: NDArray, noise_std: float) -> NDArray:
    """Analytic gradient of log p_sigma(y) for 1-D Gaussian mixture observed under Gaussian noise."""
    y = np.atleast_1d(y).astype(np.float64)
    obs_var = sigmas ** 2 + noise_std ** 2
    obs_std = np.sqrt(obs_var)
    components = (
        weights[None, :] *
        np.exp(-0.5 * ((y[:, None] - mus[None, :]) / obs_std[None, :]) ** 2) /
        (np.sqrt(2 * np.pi) * obs_std[None, :])
    )
    p_y = components.sum(axis=1, keepdims=True)
    grad_components = -components * (y[:, None] - mus[None, :]) / obs_var[None, :]
    return (grad_components.sum(axis=1, keepdims=True) / p_y).squeeze()


# Bimodal prior
mus = np.array([-3.0, +3.0])
sigmas = np.array([0.3, 0.3])
weights = np.array([0.5, 0.5])

noise_std = 1.0
y_grid = np.linspace(-7, 7, 401)

denoiser = gmm_mmse_denoiser(y_grid, mus, sigmas, weights, noise_std)
residual = denoiser - y_grid
score = gmm_score(y_grid, mus, sigmas, weights, noise_std)
tweedie_rhs = noise_std ** 2 * score

max_err = np.abs(residual - tweedie_rhs).max()
print(f'Maximum |denoiser-residual - sigma^2 * score| = {max_err:.2e}')
print('Should be ~0 (numerical precision) — Tweedie identity is exact.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(y_grid, gmm_pdf(y_grid, mus, sigmas, weights), label=r'$p(x)$ prior', color='tab:blue')
p_sigma = np.array([(weights * np.exp(-0.5 * (y - mus) ** 2 / (sigmas ** 2 + noise_std ** 2))
                     / np.sqrt(2 * np.pi * (sigmas ** 2 + noise_std ** 2))).sum() for y in y_grid])
axes[0].plot(y_grid, p_sigma, label=fr'$p_\sigma(y)$ ($\sigma$={noise_std})', color='tab:orange')
axes[0].set_title('Prior $p(x)$ vs noisy observation density $p_\\sigma(y)$')
axes[0].legend(); axes[0].set_xlabel('y'); axes[0].set_ylabel('density')

axes[1].plot(y_grid, residual, label=r'denoiser residual $\hat x(y) - y$', color='tab:green', linewidth=2.5)
axes[1].plot(y_grid, tweedie_rhs, '--', label=r'$\sigma^2 \nabla_y \log p_\sigma(y)$ (Tweedie RHS)',
             color='red', linewidth=1.5)
axes[1].set_title('Tweedie/Miyasawa identity verification')
axes[1].legend(); axes[1].set_xlabel('y'); axes[1].set_ylabel('value')
axes[1].axhline(0, color='gray', linewidth=0.5)
plt.tight_layout(); plt.show()

**Verification**: 두 곡선이 시각적으로 *완전히 일치* — Tweedie 항등식 $f(y) = \sigma^2 \nabla_y \log p_\sigma(y)$이 numerical precision 수준에서 정확하다.

The two curves overlap perfectly — confirming the Tweedie/Miyasawa identity.

---

## Part 2: Stochastic ascent on prior (Algorithm 1) / 사전분포에서 확률적 ascent

Random init $y_0 \sim \mathcal N(0, \sigma_0^2)$ (centered between modes), iterate Eq. 5:
$$
y_t = y_{t-1} + h_t f(y_{t-1}) + \gamma_t z_t,
$$
with $\gamma_t$ from Eq. 8 and $h_t$ from the schedule. Convergence to one of the two modes ($\pm 3$).

1차원 bimodal prior에서 random init부터 stochastic ascent로 mode에 수렴하는지 확인.

In [ ]:
def stochastic_ascent_1d(
    f_fn,
    y0: float,
    sigma_0: float = 1.0,
    sigma_L: float = 0.05,
    h0: float = 0.05,
    beta: float = 0.5,
    max_iter: int = 200,
    seed: int = 0,
) -> tuple[NDArray, NDArray]:
    """Algorithm 1 of Kadkhodaie-Simoncelli for 1-D scalar problem.

    Args:
        f_fn: callable y -> denoiser_residual = sigma^2 * score, with sigma adapted on the fly.
              Here we pass a lambda that uses the current effective sigma_t.
        y0: scalar initialization.
        sigma_0: initial effective noise std.
        sigma_L: terminal effective noise std.
        h0: initial step fraction.
        beta: injected-noise factor (1 = no noise; 0 = full noise).
        max_iter: maximum iterations.
        seed: RNG seed.

    Returns:
        Trajectory of y values and effective sigma values.
    """
    g = np.random.default_rng(seed)
    ys = [y0]
    sigmas_eff = [sigma_0]
    sigma_prev = sigma_0
    for t in range(1, max_iter + 1):
        if sigma_prev <= sigma_L:
            break
        h_t = h0 * t / (1 + h0 * (t - 1))
        d_t = f_fn(ys[-1], sigma_prev)
        sigma_t_sq = max(d_t ** 2, 1e-12)  # 1-D so ||d||^2/N = d^2
        gamma_t_sq = max(((1 - beta * h_t) ** 2 - (1 - h_t) ** 2) * sigma_t_sq, 0.0)
        z_t = g.standard_normal()
        y_new = ys[-1] + h_t * d_t + np.sqrt(gamma_t_sq) * z_t
        sigma_prev = np.sqrt((1 - beta * h_t) ** 2 * sigma_t_sq)
        ys.append(y_new)
        sigmas_eff.append(sigma_prev)
    return np.array(ys), np.array(sigmas_eff)


# f(y) = sigma^2 * score, where sigma is the *current* effective noise std
def f_function(y: float, sigma: float) -> float:
    arr_y = np.array([y])
    return float(gmm_mmse_denoiser(arr_y, mus, sigmas, weights, sigma)[0] - y)


fig, axes = plt.subplots(1, 2, figsize=(12, 4))
trajectories = []
for seed in range(8):
    y0 = rng.normal(0.0, 1.0)
    traj, _ = stochastic_ascent_1d(f_function, y0, sigma_0=1.0, sigma_L=0.05,
                                     h0=0.05, beta=0.5, max_iter=200, seed=seed)
    trajectories.append(traj)
    axes[0].plot(traj, alpha=0.7)
axes[0].axhline(+3, color='red', linestyle='--', label='mode +3')
axes[0].axhline(-3, color='blue', linestyle='--', label='mode -3')
axes[0].set_xlabel('iteration'); axes[0].set_ylabel('y')
axes[0].set_title('Trajectories of stochastic ascent (8 random inits)')
axes[0].legend()

# Histogram of converged values
final_values = np.array([traj[-1] for traj in trajectories])
axes[1].hist(final_values, bins=20, color='tab:purple', edgecolor='black', alpha=0.7)
axes[1].axvline(+3, color='red', linestyle='--')
axes[1].axvline(-3, color='blue', linestyle='--')
axes[1].set_xlabel('final y'); axes[1].set_ylabel('count')
axes[1].set_title('Final values cluster near modes ±3')
plt.tight_layout(); plt.show()

print(f'Final values from 8 inits: {final_values}')
print(f'Mean abs distance to nearest mode: {np.abs(np.abs(final_values) - 3).mean():.3f}')

---

## Part 3: Synthetic 2-D image setup for inpainting / 인페인팅용 2D 합성 영상

We build a 2-D analogue for the inpainting demo. Use a *piecewise-constant* image (smooth Gaussian prior won't show interesting inpainting behaviour). Our "prior" is implicit via a simple Gaussian-smoothing denoiser — though not optimal, it embeds a smooth-image prior usable for projecting onto the manifold of low-frequency images.

2D 합성 영상에서 inpainting을 시연. 단순 Gaussian smoothing을 "디노이저"로 사용 — *low-frequency manifold* 위로의 projection. 진짜 자연 영상 prior는 아니지만 algorithm의 동작 원리를 보여준다.

In [ ]:
from scipy.ndimage import gaussian_filter


def make_synthetic_image(size: int = 32) -> NDArray[np.float64]:
    """Make a smooth synthetic image: Gaussian blob on white background."""
    yy, xx = np.meshgrid(np.linspace(-1, 1, size), np.linspace(-1, 1, size), indexing='ij')
    img = 0.7 * np.exp(-(xx ** 2 + yy ** 2) / 0.3 ** 2)  # central Gaussian blob
    img += 0.5 * np.exp(-((xx - 0.5) ** 2 + (yy + 0.5) ** 2) / 0.15 ** 2)  # secondary blob
    img = np.clip(img, 0.0, 1.0)
    return img


def gaussian_denoiser(y: NDArray[np.float64], sigma_noise: float) -> NDArray[np.float64]:
    """Cheap "denoiser": Gaussian smoothing with bandwidth tied to sigma_noise.

    For a Gaussian-smoothness prior with bandwidth tau, MMSE denoiser is approximately
    Gaussian filtering with width sqrt(sigma^2 / (sigma^2 + tau^2)) * tau. We use a simple
    tau-proportional bandwidth.
    """
    bandwidth = max(0.5 * sigma_noise / 0.2, 0.3)
    return gaussian_filter(y, sigma=bandwidth, mode='reflect')


clean_img = make_synthetic_image(size=32)

# Set up inpainting: mask out a square region (10x10 in the middle)
mask = np.ones_like(clean_img)  # 1 = retained, 0 = missing
mask[12:22, 12:22] = 0.0
observed_pixels = clean_img * mask

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
axes[0].imshow(clean_img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Ground truth / 정답')
axes[1].imshow(mask, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Mask (1=keep)\nmissing pixels: {int((1-mask).sum())}')
axes[2].imshow(observed_pixels, cmap='gray', vmin=0, vmax=1)
axes[2].set_title('Observation $M^Tx$ / 측정값')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()
print(f'Image shape: {clean_img.shape}, missing block: {int((1-mask).sum())} pixels.')

---

## Part 4: Constrained sampling for inpainting (Algorithm 2) / 인페인팅을 위한 제약 샘플링

Use Eq. 9: $d_t = (I - MM^T) f(y_{t-1}) + M(x^c - M^T y_{t-1})$.

For pixel-mask inpainting with $M$ a subset of identity columns, $MM^T = \text{diag}(\text{mask})$, so:
$$
d_t = (1 - \text{mask})\odot f(y_{t-1}) + \text{mask}\odot(\text{observed} - y_{t-1}).
$$
Prior gradient on missing pixels, measurement pull on retained pixels.

Eq. 9 인페인팅용 단순화: $d_t$ = 누락 픽셀에서는 prior gradient, 측정 픽셀에서는 측정값 fidelity.

In [ ]:
def constrained_sampling_inpaint(
    observed: NDArray[np.float64],
    mask: NDArray[np.float64],
    denoiser_fn,
    sigma_0: float = 0.6,
    sigma_L: float = 0.05,
    h0: float = 0.05,
    beta: float = 0.5,
    max_iter: int = 100,
    seed: int = 0,
) -> tuple[NDArray[np.float64], list[float]]:
    """Algorithm 2 specialised for pixel-mask inpainting.

    Args:
        observed: image with missing pixels = 0 (shape HxW).
        mask: 0/1 array, 1 = retained pixel.
        denoiser_fn: callable (y, sigma) -> denoised image.
        sigma_0, sigma_L, h0, beta: schedule parameters.
        max_iter: maximum iterations.
        seed: RNG seed.

    Returns:
        Final image and list of effective sigmas per iteration.
    """
    g = np.random.default_rng(seed)
    # Initialisation: observed pixels = observed, missing = 0.5 + Gaussian noise
    y = mask * observed + (1 - mask) * (0.5 + sigma_0 * g.standard_normal(observed.shape))
    sigma_prev = sigma_0
    sigma_history = [sigma_prev]
    n_pix = observed.size
    for t in range(1, max_iter + 1):
        if sigma_prev <= sigma_L:
            break
        h_t = h0 * t / (1 + h0 * (t - 1))
        residual = denoiser_fn(y, sigma_prev) - y  # f(y) = sigma^2 * score
        # Eq. 9 specialised:
        d_t = (1 - mask) * residual + mask * (observed - y)
        sigma_t_sq = max((d_t ** 2).sum() / n_pix, 1e-12)
        gamma_t_sq = max(((1 - beta * h_t) ** 2 - (1 - h_t) ** 2) * sigma_t_sq, 0.0)
        z = g.standard_normal(observed.shape)
        y = y + h_t * d_t + np.sqrt(gamma_t_sq) * z
        sigma_prev = np.sqrt(max((1 - beta * h_t) ** 2 * sigma_t_sq, 1e-12))
        sigma_history.append(sigma_prev)
    return y, sigma_history


# Run constrained sampling
restored, sigma_hist = constrained_sampling_inpaint(
    observed_pixels, mask, gaussian_denoiser,
    sigma_0=0.6, sigma_L=0.02, h0=0.05, beta=0.5, max_iter=200, seed=1
)

# PSNR helper
def psnr(a, b):
    mse = ((a - b) ** 2).mean()
    return float('inf') if mse == 0 else 10 * np.log10(1.0 / mse)

psnr_obs = psnr(observed_pixels, clean_img)  # treats missing block as 0
psnr_rest = psnr(restored, clean_img)
print(f'PSNR observation (missing as 0): {psnr_obs:.2f} dB')
print(f'PSNR restoration: {psnr_rest:.2f} dB')
print(f'PSNR improvement on missing block only: {psnr(restored * (1-mask), clean_img * (1-mask)):.2f} dB')

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(clean_img, cmap='gray', vmin=0, vmax=1); axes[0].set_title('Ground truth')
axes[1].imshow(observed_pixels, cmap='gray', vmin=0, vmax=1); axes[1].set_title('Observed (with hole)')
axes[2].imshow(restored, cmap='gray', vmin=0, vmax=1)
axes[2].set_title(f'Restored (Algo 2)\nPSNR={psnr_rest:.2f} dB')
axes[3].imshow(restored - clean_img, cmap='RdBu', vmin=-0.3, vmax=0.3); axes[3].set_title('Residual error')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

---

## Part 5: Multiple stochastic samples / 여러 확률적 샘플

Different RNG seeds → different reconstructions all consistent with the measurements (Fig. 3 in paper).

다른 random seed로 sampling하면 다른 plausible 복원 결과 생성 — 측정 부분공간 ∩ manifold의 다른 점들.

In [ ]:
n_samples = 4
samples = []
for s in range(n_samples):
    out, _ = constrained_sampling_inpaint(
        observed_pixels, mask, gaussian_denoiser,
        sigma_0=0.6, sigma_L=0.02, h0=0.05, beta=0.3, max_iter=200, seed=100 + s
    )
    samples.append(out)

samples_arr = np.stack(samples, axis=0)
mean_sample = samples_arr.mean(axis=0)

fig, axes = plt.subplots(1, n_samples + 2, figsize=(15, 3))
axes[0].imshow(clean_img, cmap='gray', vmin=0, vmax=1); axes[0].set_title('Truth')
axes[0].axis('off')
for i, s in enumerate(samples):
    axes[i + 1].imshow(s, cmap='gray', vmin=0, vmax=1)
    axes[i + 1].set_title(f'sample {i}\nPSNR={psnr(s, clean_img):.1f}dB'); axes[i + 1].axis('off')
axes[-1].imshow(mean_sample, cmap='gray', vmin=0, vmax=1)
axes[-1].set_title(f'mean of {n_samples}\nPSNR={psnr(mean_sample, clean_img):.1f}dB'); axes[-1].axis('off')
plt.tight_layout(); plt.show()

print('Each sample is slightly different (different injected noise) but all consistent with observed pixels.')
print("Averaging samples is a 'convex combination' — better PSNR but blurrier (paper §3.2 obs).")

---

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | This Notebook / 본 노트북 | Modern Equivalent / 현대 동등물 |
|---|---|---|---|
| Tweedie/Miyasawa identity | Eq. 3 ($\hat x = y + \sigma^2 \nabla \log p_\sigma$) | Part 1 — analytic GMM verification | All score-based diffusion samplers |
| Stochastic ascent (Algo 1) | Eq. 5 with adaptive $\gamma_t$ | Part 2 — 1-D bimodal sampling | DDPM/DDIM reverse step |
| Constrained sampling (Algo 2) | Eq. 9 with projection | Part 3–4 — 2-D inpainting | DDRM (paper #26), DPS, DDNM |
| Multi-sample diversity | Fig. 3, 5 | Part 5 — 4 random-seed samples | Posterior sampling for ill-posed inverse |

### Connections to other papers / 다른 논문과의 연결

- **Paper #26 (DDRM)**: extends this notebook's *constrained sampling* idea to large pretrained DDPM via SVD-based spectral decomposition.
- **Paper #27 (Cold Diffusion)**: challenges the noise-centric framework here — same algorithmic skeleton works without Gaussian noise.
- **Paper #16 (Noise2Noise)**: provides the means to *train* the underlying denoiser without clean data — combine with this paper for fully self-supervised inverse problem solving.
- **Paper #1 (Donoho-Johnstone wavelet thresholding)**: the historical predecessor — soft-thresholding as the simplest possible MMSE denoiser; this paper generalises to arbitrary CNN denoisers.

### Take-away / 결론

본 노트북은 Kadkhodaie-Simoncelli의 세 핵심을 직접 검증했다:
1. **이론**: Tweedie 항등식이 numerical precision 수준에서 정확함 (Part 1).
2. **Synthesis**: bimodal prior에서 stochastic ascent가 mode로 수렴 (Part 2).
3. **Inverse problems**: constrained sampling이 inpainting을 풀고, multi-sample diversity가 측정의 ill-posedness를 직관적으로 표현 (Parts 3–5).

Verifies the three Kadkhodaie-Simoncelli ideas: (1) Tweedie identity is exact at machine precision; (2) stochastic ascent converges to prior modes from random init; (3) constrained sampling gives multiple plausible inpaintings consistent with the observed pixels.

이 단순한 demo가 다음 1년 내에 DDRM (paper #26)으로 확장되어 사전학습 DDPM과 결합되며 ImageNet 4× SR에서 26.55 dB를 달성한다 — 같은 아이디어가 그저 *더 큰 디노이저*로 옮겨갔을 뿐이다.

This simple demo is extended within a year (paper #26, DDRM) by combining the same conditional-sampling idea with pretrained DDPMs, reaching 26.55 dB on ImageNet 4× SR — the algorithm is unchanged; only the denoiser scales up.